# Draft strategy

What roster construction is actually worth in **these two leagues**, as live queries. Re-run the
notebook and the numbers update — nothing below is transcribed by hand.

The question that started this: a "full house" opening — **three running backs and two receivers in
the first five rounds** — on the theory that running back is the scarce position and that even elite
receivers can't carry a team the way an elite back can.

The full house loses, and it now clears the significance bar as a *bad* opening in both leagues
(§6). But the more useful answer is that **the two leagues no longer want the same draft at all.**
Sleeper changed in August 2026: 12 teams became 14, and one of its two flex spots became a
**SUPER_FLEX**. ESPN did not change. So one league now starts a quarterback-eligible flex and the
other doesn't, and that single slot outweighs everything the original question was about.

- **ESPN (1QB)** — the unexciting answer holds. No position's slope clears significance, opening
  composition is worth about one good waiver claim, and the reliable signal is which openings
  *lose*.
- **Sleeper (superflex)** — composition becomes the dominant decision. The quarterback slope is the
  largest effect anywhere in this notebook, every opening in the top ten takes at least one
  quarterback and the top seven take two, and openings that ignore the position — the full house
  among them — are significantly bad.

Section 8 is where that gets dangerous rather than merely interesting. The ADP board these drafts
run off is a **1QB board**, priced by sites whose users mostly play one quarterback. Drafting
best-available off it in a superflex league isn't discipline, it is systematically declining to fill
the new slot — and it costs Sleeper more than any composition mistake in the sweep.

Sections 2-4 measure the scarcity premise the full house rests on, section 5 simulates ~89,000
drafts, sections 6-8 decide how much of the result to believe, and section 9 isolates what the
superflex slot itself is doing by running *both* leagues in *both* formats.

**Contents**
1. [The two leagues](#leagues)
2. [Is the premise true? Positional scarcity](#scarcity)
3. [What each round actually returned](#rounds)
4. [Where mid-round running backs go wrong](#hitrate)
5. [The simulation](#simulation)
6. [Is any of it significant?](#significance)
7. [Ordering: RB early vs RB often](#ordering)
8. [Who else is at the table](#field)
9. [What the superflex slot is doing](#superflex)
10. [What to actually do](#doing)
11. [The 2026 board](#board)

In [1]:
# Find the repo root from wherever the kernel started, so `src` imports work.
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats

from src.query import q, tables, columns, peek

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

tables("gold").query("table.str.startswith('draft_strategy')")

,table,layer,rows
9,draft_strategy_results,gold,88704
10,draft_strategy_summary,gold,376


<a id="leagues"></a>
## 1. The two leagues

Every section below is computed twice, once per league, because these two leagues now answer the
same question differently enough that a single recommendation would be wrong in one of them. Both
rows come from `league_settings`, which reads each platform's own configuration rather than anything
typed in by hand.

- **Sleeper** — 14 teams, half-PPR, one flex **and one superflex**: 8 skill starters per team, so
  112 skill players start every week.
- **ESPN** — 10 teams, full PPR, one flex, no superflex: 7 skill starters, so only 70 do.

Three consequences to carry forward. The first two set up section 2; the third is the one that ends
up mattering most:

- **Pool depth sets replacement level.** Sleeper starts 112 skill players to ESPN's 70. A position
  is only scarce relative to what you could have had instead, so the deeper league makes the
  premium at the top of a position worth more, not less.
- **Full PPR pays receivers.** ESPN's point per reception lifts the position that catches the most
  passes — precisely the one the full-house theory wants to fade.
- **Sleeper's superflex slot makes quarterback a two-starter position.** ESPN's does not. This is
  the difference the rest of the notebook keeps running into, and section 9 measures it directly.

**A note on how that last line is derived.** An earlier version of this notebook asserted that
neither league was superflex and hard-coded that assumption into every query. Sleeper converted a
flex spot in August 2026 and the assumption rotted silently — the queries kept returning rows, just
the wrong ones. So the cell below reads each league's real format out of `league_settings` into
`ACTUAL` and everything downstream filters on that. Section 9 re-checks it against the platforms'
raw settings.

In [2]:
settings = q("""
    SELECT
        league_key                          AS league,
        team_count                          AS teams,
        rec_pts                             AS "pts/reception",
        qb_slots || ' QB'                   AS qb,
        rb_slots || ' RB'                   AS rb,
        wr_slots || ' WR'                   AS wr,
        te_slots || ' TE'                   AS te,
        flex_slots || ' FLEX'               AS flex,
        superflex_slots || ' SUPERFLEX'     AS superflex,
        bench_slots                         AS bench,
        qb_slots + rb_slots + wr_slots + te_slots + flex_slots + superflex_slots
                                            AS "skill starters",
        team_count * (qb_slots + rb_slots + wr_slots + te_slots + flex_slots + superflex_slots)
                                            AS "skill players starting"
    FROM league_settings
    ORDER BY league_key
""").set_index("league").T

# Which simulator variant each league *is*, derived from the settings rather than hard-coded.
# This notebook used to assume "neither league is superflex" and filter `variant = 'actual'`
# everywhere. Sleeper went superflex, and every one of those filters was quietly reading the
# wrong rows with no error to show for it. Deriving it means the settings changing again is a
# number that moves, not a claim that silently rots.
ACTUAL = dict(
    q("""
        SELECT league_key,
               CASE WHEN superflex_slots > 0 THEN 'superflex' ELSE '1qb' END AS variant
        FROM league_settings
    """).itertuples(index=False, name=None)
)
print("real format per league:", ACTUAL, "\n")

settings

real format per league: {'sleeper': 'superflex', 'espn': '1qb'} 



league,espn,sleeper
teams,10,14
pts/reception,1.0,0.5
qb,1 QB,1 QB
rb,2 RB,2 RB
wr,2 WR,2 WR
te,1 TE,1 TE
flex,1 FLEX,1 FLEX
superflex,0 SUPERFLEX,1 SUPERFLEX
bench,7,5
skill starters,7,8


<a id="scarcity"></a>
## 2. Is the premise true? Positional scarcity

**In one league, deeply. In the other, barely at all.** The full house's premise is a claim about a
roster format, not about football, and these two formats disagree about it.

The table below is points over replacement by *realised* positional finish, averaged over 2015-2025,
in each league's own scoring. `RB-WR` is the whole argument in one column: how much more the RB
finishing Nth was worth than the WR finishing Nth. Read where it crosses zero — above the crossover
running back is the scarcer asset and the premise holds; below it the premise inverts and receivers
are worth *more* at the same rank.

The second cell computes that crossover, and the two leagues are nowhere near each other. Section
1's first two consequences are why: ESPN's shallower pool pushes replacement level high enough that
the RB curve has already flattened into it, and full PPR lifts the receiver curve to meet it, while
Sleeper's 14 teams and half-PPR do neither. Sleeper's crossover sits further out under the new
settings than it did at 12 teams, in the direction the pool-depth argument predicts — though those
settings changed the superflex slot at the same time, so this isn't a clean read of team count alone.

So running back scarcity is real in Sleeper and marginal in ESPN. Note what that does and does not
license: it is an argument for taking a back early in Sleeper, not for taking three — and sections
5-6 will show it is comprehensively outranked there by a position this table treats as an
afterthought.

`starters_at_position` in `points_over_replacement` is where format enters the arithmetic. It counts
dedicated slots plus the flex spots each position actually won, which is what pushes replacement
level down in the deeper league — and, in Sleeper, what now lets quarterbacks claim a starting slot
they previously couldn't.

In [3]:
def por_curve(league_key, max_rank=30):
    return q("""
        SELECT position_rank AS rank,
               MAX(CASE WHEN position = 'RB' THEN por END) AS RB,
               MAX(CASE WHEN position = 'WR' THEN por END) AS WR,
               MAX(CASE WHEN position = 'TE' THEN por END) AS TE,
               MAX(CASE WHEN position = 'QB' THEN por END) AS QB
        FROM (
            SELECT position, position_rank, AVG(points_over_replacement) AS por
            FROM points_over_replacement
            WHERE league_key = ? AND season <= (SELECT MAX(season) FROM weekly_stats)
            GROUP BY position, position_rank
        )
        GROUP BY position_rank
        HAVING position_rank <= ?
        ORDER BY position_rank
    """, [league_key, max_rank])


curves = {}
for league_key in ("sleeper", "espn"):
    curve = por_curve(league_key).set_index("rank")
    curve["RB-WR"] = curve.RB - curve.WR
    curves[league_key] = curve

pd.concat(curves, axis=1).round(1)

sleeper                              espn                           
          RB     WR     TE     QB RB-WR     RB     WR     TE     QB RB-WR
rank                                                                     
1      217.9  174.1  117.7  259.3  43.9  205.3  181.8  121.7  114.3  23.5
2      176.0  147.1   85.9  220.8  28.8  153.1  145.4   86.9   76.0   7.7
3      159.6  129.8   69.1  211.3  29.7  137.4  130.8   63.4   67.0   6.5
4      146.0  116.0   60.0  200.4  30.0  119.2  114.4   51.2   54.3   4.9
5      128.3  109.1   50.4  185.4  19.1  102.6  103.4   40.2   39.0  -0.8
6      118.3   99.9   42.0  171.2  18.4   92.5   94.5   31.0   26.7  -2.0
7      105.5   89.5   34.1  164.8  16.0   75.1   82.8   18.4   20.7  -7.8
8       95.2   83.5   28.9  160.1  11.7   66.0   75.3   14.6   15.4  -9.3
9       87.0   79.5   25.4  154.5   7.5   58.9   69.2   10.6   11.0 -10.3
10      83.1   75.1   19.5  150.4   8.0   53.5   63.0    3.3    5.5  -9.5
11      78.3   70.5   13.2  145.2   7.8   47.0   60.3    0.0    0.0 -13.3
12      72.1   68.2   10.2  135.6   3.9   41.9   56.4   -5.3   -9.9 -14.5
13      68.9   65.6    7.8  125.3   3.3   36.3   52.2  -10.4  -22.4 -15.8
14      65.9   63.5    3.8  119.2   2.4   32.2   49.4  -16.7  -25.5 -17.2
15      60.1   57.5    0.0  111.6   2.6   27.5   46.2  -20.0  -32.8 -18.7
16      55.6   53.7   -3.0  106.0   1.9   24.6   40.1  -24.3  -37.5 -15.5
17      52.0   50.8   -6.0   99.8   1.2   20.6   36.4  -27.0  -45.5 -15.8
18      48.8   49.2   -7.7   91.8  -0.4   15.5   32.9  -30.3  -53.9 -17.4
19      45.9   47.0  -10.4   88.5  -1.2   12.3   30.5  -34.1  -58.8 -18.2
20      42.0   43.8  -13.6   80.7  -1.8    8.9   28.0  -36.5  -64.4 -19.1
21      39.0   42.4  -17.8   70.8  -3.4    5.5   26.6  -40.6  -73.8 -21.0
22      36.2   40.9  -19.8   61.2  -4.7    0.5   22.9  -42.9  -83.9 -22.3
23      32.2   38.6  -21.8   52.8  -6.4   -3.0   19.2  -47.2  -92.2 -22.2
24      27.4   35.2  -25.0   46.8  -7.8   -7.8   17.4  -50.2  -97.4 -25.1
25      22.9   32.5  -27.4   37.1  -9.6  -11.0   13.4  -54.4 -107.7 -24.5
26      19.0   29.7  -30.7   30.0 -10.6  -14.6   10.8  -58.7 -112.7 -25.4
27      16.7   27.1  -33.6   22.7 -10.3  -17.5    7.4  -62.3 -122.3 -24.9
28      12.5   25.3  -36.1   11.8 -12.8  -20.9    4.8  -64.4 -132.2 -25.6
29       8.7   22.9  -38.1   -0.5 -14.2  -26.1    2.2  -65.7 -143.5 -28.3
30       7.1   22.1  -40.2  -13.4 -15.0  -28.8    0.2  -68.1 -157.0 -29.0

In [4]:
# Where does the running back premium run out? First rank at which the WR finishing there is worth
# more than the RB finishing there.
for league_key, curve in curves.items():
    ahead = curve.index[curve["RB-WR"] > 0]
    crossover = int(ahead.max()) + 1 if len(ahead) else 1
    top2 = curve.loc[:2, "RB-WR"].mean()
    print(
        f"{league_key:>8}: RB is worth more than WR through rank {crossover - 1}, "
        f"WR from rank {crossover} on. "
        f"Premium at the top (ranks 1-2): {top2:+.0f} pts."
    )

 sleeper: RB is worth more than WR through rank 17, WR from rank 18 on. Premium at the top (ranks 1-2): +36 pts.
    espn: RB is worth more than WR through rank 4, WR from rank 5 on. Premium at the top (ranks 1-2): +16 pts.


<a id="rounds"></a>
## 3. What each round actually returned

Section 2 measured *finishes*. This one measures **picks**, which is the thing a strategy actually
controls: for every player with a preseason consensus ADP since 2015, what did a pick in that round
go on to return? Value is `draft_value.actual_value` — points over replacement in that league's
scoring, floored at zero, because nobody is forced to start a player worse than the waiver wire — and
a drafted player who never played a snap is carried as a genuine zero rather than dropped, so the
bust rate is priced in.

Rounds are that league's own team count, so "round 3" now means picks 29-42 in Sleeper and 21-30 in
ESPN.

Section 2's split survives the translation into draft picks, which is the point of running it twice:
the second cell reports the rounds in which running back actually out-returned receiver, and the two
leagues do not agree about whether any exist. Where the premium is real it is concentrated early —
by round 3 the position that was scarce at the top has stopped being the better buy in either league.

That matters for the original question because rounds 3-5 are exactly where a full-house opening
spends its third running back.

In [5]:
def value_by_round(league_key, rounds=8):
    return q("""
        SELECT CAST(CEIL(d.consensus_adp / l.team_count) AS INT) AS round,
               AVG(CASE WHEN position = 'RB' THEN actual_value END) AS RB,
               AVG(CASE WHEN position = 'WR' THEN actual_value END) AS WR,
               AVG(CASE WHEN position = 'TE' THEN actual_value END) AS TE,
               AVG(CASE WHEN position = 'QB' THEN actual_value END) AS QB,
               COUNT(*) AS picks
        FROM draft_value d
        JOIN league_settings l ON l.league_key = d.league_key
        WHERE d.league_key = ? AND d.actual_value IS NOT NULL
          AND d.consensus_adp <= l.team_count * ?
        GROUP BY 1 ORDER BY 1
    """, [league_key, rounds])


returns = {k: value_by_round(k).set_index("round") for k in ("sleeper", "espn")}
pd.concat(returns, axis=1).round(1)

sleeper                           espn                         
           RB    WR    TE     QB picks    RB    WR     TE    QB picks
round                                                                
1        96.0  88.9  70.5    6.7   146  79.6  80.8   83.9   NaN    98
2        68.1  55.3  65.3  182.2   152  61.8  71.5  105.6  19.0   119
3        45.5  45.8  59.8  159.3   165  33.8  35.0   37.5  61.9    99
4        26.7  43.2  38.6  123.9   143  25.8  41.9   55.4  40.4   119
5        28.6  27.1  23.3  159.6   158  14.6  37.9   36.2  19.0   115
6        23.9  18.0  14.0  104.6   146  16.4  23.0   19.2  27.0   102
7        14.0  17.6  10.4  116.9   150  11.8  20.8   24.6  29.0   112
8        16.9  14.9  23.4  131.5   136   8.4  12.3    7.5   6.4   103

In [6]:
# Which position returned more, per round, per league.
for league_key, table in returns.items():
    rb_rounds = table.index[table.RB > table.WR].tolist()
    print(f"{league_key:>8}: RB out-returned WR in round(s) {rb_rounds or 'none'} "
          f"of the first {len(table)}")

 sleeper: RB out-returned WR in round(s) [1, 2, 5, 6, 8] of the first 8
    espn: RB out-returned WR in round(s) none of the first 8


<a id="hitrate"></a>
## 4. Where mid-round running backs go wrong

The averages in section 3 hide *how* they happen, and the how is the useful part. `boom_bust`
classifies every drafted player-season by whether he finished as a top-`team_count` asset at his
position — an absolute measure, not "did he beat his ADP" — so this is the hit rate on a pick.

Rounds 1-3 look similar for both positions. The separation is in **rounds 4-5**, where a running
back taken there hits markedly less often than a receiver taken there, in both leagues — around
half as often in Sleeper. After two
sections of the leagues disagreeing, this is the one place they agree — and it is the sharpest
evidence against the full house that doesn't require the simulator.

The issue isn't that the mid-round running back is a bad player; it's that he is the least reliable
pick on the board, and a composition fixed before the draft starts is a commitment to take one
anyway.

`n` columns are shown because the tight end row is thin — few tight ends go early enough to appear —
and shouldn't be read as a finding.

In [7]:
def hit_rate(league_key, rounds=6):
    return q("""
        SELECT CAST(CEIL(b.consensus_adp / l.team_count) AS INT) AS round,
               100 * AVG(CASE WHEN position = 'RB' THEN is_elite_finish::INT END) AS "RB hit%",
               100 * AVG(CASE WHEN position = 'WR' THEN is_elite_finish::INT END) AS "WR hit%",
               SUM(CASE WHEN position = 'RB' THEN 1 ELSE 0 END) AS "n RB",
               SUM(CASE WHEN position = 'WR' THEN 1 ELSE 0 END) AS "n WR",
               100 * AVG(CASE WHEN position = 'TE' THEN is_elite_finish::INT END) AS "TE hit%",
               100 * AVG(CASE WHEN position = 'QB' THEN is_elite_finish::INT END) AS "QB hit%"
        FROM boom_bust b
        JOIN league_settings l ON l.league_key = b.league_key
        WHERE b.league_key = ? AND b.consensus_adp <= l.team_count * ?
        GROUP BY 1 ORDER BY 1
    """, [league_key, rounds])


hits = {k: hit_rate(k).set_index("round") for k in ("sleeper", "espn")}
pd.concat(hits, axis=1).round(1)

sleeper                                        espn                                    
      RB hit% WR hit%  n RB  n WR TE hit% QB hit% RB hit% WR hit%  n RB  n WR TE hit% QB hit%
round                                                                                        
1        64.3    66.1  84.0  56.0    75.0     0.0    50.8    55.9  61.0  34.0   100.0     NaN
2        54.4    48.5  57.0  68.0    81.8    80.0    48.1    51.9  54.0  52.0    83.3    66.7
3        36.1    31.4  61.0  70.0    86.7    80.0    35.3    17.8  34.0  45.0    50.0    80.0
4         9.8    33.8  41.0  65.0    62.5    57.1    25.6    28.0  43.0  50.0    70.0    66.7
5        16.7    18.6  54.0  59.0    52.6    70.8     8.6    27.3  35.0  55.0    72.7    42.9
6        14.3    11.3  49.0  53.0    52.4    42.1    11.4    10.8  35.0  37.0    38.5    56.2

In [8]:
for league_key, table in hits.items():
    late = table.loc[4:5]
    print(f"{league_key:>8}: rounds 4-5 hit rate — RB {late['RB hit%'].mean():.1f}% "
          f"vs WR {late['WR hit%'].mean():.1f}%")

 sleeper: rounds 4-5 hit rate — RB 13.2% vs WR 26.2%
    espn: rounds 4-5 hit rate — RB 17.1% vs WR 27.6%


<a id="simulation"></a>
## 5. The simulation

Sections 2-4 are circumstantial: they price positions and picks, not rosters. A draft strategy is a
claim about *roster construction*, so `src/gold/draft_strategy.py` runs the draft — ~89,000 times.

Each simulated draft is a snake draft off that season's real `adp_consensus` board, with every team
scored on the hindsight-best starting lineup it could field from the roster it ended up with, using
that league's own scoring and starting slots. One team — the focal team — follows a strategy for the
first five rounds; everyone else takes the best player left by ADP. The focal seat then rotates
through every draft slot, and the whole thing repeats for every season from 2015 on.

A **strategy** is a *composition*: how many of each position in the first five rounds, with ADP
deciding the order. That is what a human actually does — nobody committed to "3 RB, 2 WR" passes the
top receiver on the board at 1.03 to force a back — and it keeps the strategy space at 36 rather than
4⁵ = 1024. `ADP` is the control: no constraint at all, best available for every round.

Every league is run in **both** formats, `1qb` and `superflex`, and each league's own row in
`league_settings` decides which of the two is real for it — `ACTUAL` from section 1. The tables below
show each league in its real format; section 9 uses the other one.

`points_vs_field` is the focal team's starting-lineup total minus the average of the other teams in
that same draft, so the control sits at exactly zero against a pure-ADP field by construction. Its
`t_stat` is `NaN` for the same reason: with every team drafting the same way the focal team *is* the
field, and there is no effect there to test. Full details and caveats are in the module docstring.

**This section ranks; it does not conclude.** Thirty-seven strategies sorted by an eleven-season mean
will always have a top row. Section 6 decides which of these gaps are real, and this time — unlike
the version of this notebook written before Sleeper changed — the answer is not the same in both
leagues.

In [9]:
q("""
    SELECT league_key AS league, variant, field_model AS field,
           COUNT(*) AS drafts,
           COUNT(DISTINCT strategy) AS strategies,
           COUNT(DISTINCT season) AS seasons,
           MIN(season) AS "from", MAX(season) AS "to"
    FROM draft_strategy_results
    GROUP BY ALL ORDER BY league, variant, field
""")

,league,variant,field,drafts,strategies,seasons,from,to
0,espn,1qb,adp,6270,57,11,2015,2025
1,espn,1qb,mixed,12210,37,11,2015,2025
2,espn,superflex,adp,6270,57,11,2015,2025
3,espn,superflex,mixed,12210,37,11,2015,2025
4,sleeper,1qb,adp,8778,57,11,2015,2025
5,sleeper,1qb,mixed,17094,37,11,2015,2025
6,sleeper,superflex,adp,8778,57,11,2015,2025
7,sleeper,superflex,mixed,17094,37,11,2015,2025


In [10]:
def ranking(league_key, variant=None, field_model="adp"):
    """Composition ranking for one league. `variant` defaults to that league's real format."""
    table = q("""
        SELECT strategy, points_vs_field, finish_rank, 100 * win_rate AS "win%",
               100 * top_third_rate AS "top3rd%", t_stat, seasons_positive, n_seasons
        FROM draft_strategy_summary
        WHERE league_key = ? AND variant = ? AND field_model = ?
          AND strategy_kind <> 'ordering'
        ORDER BY points_vs_field DESC
    """, [league_key, variant or ACTUAL[league_key], field_model]).reset_index(drop=True)
    table.index = range(1, len(table) + 1)
    return table


for league_key in ("sleeper", "espn"):
    table = ranking(league_key)
    keep = pd.concat([
        table.head(5),
        table[table.strategy.isin(["ADP", "3RB2WR", "2RB3WR"])],
        table.tail(3),
    ]).drop_duplicates("strategy").sort_values("points_vs_field", ascending=False)
    print(f"\n=== {league_key} ({ACTUAL[league_key]}) — opening composition, "
          f"best to worst (of {len(table)}) ===")
    print(keep.round(2).to_string())


=== sleeper (superflex) — opening composition, best to worst (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB1RB2WR           107.70         5.32  12.99    59.09    6.93                11         11
2   2QB1RB1WR1TE            93.48         5.58  11.69    58.44    4.59                 9         11
3      2QB2RB1WR            90.01         5.71  12.34    57.14    7.34                11         11
4      2QB2RB1TE            76.07         5.84  11.69    56.49    3.35                 8         11
5         2QB3RB            72.50         5.99   9.74    51.30    4.42                10         11
20           ADP            -0.00         7.50   7.14    35.71     NaN                 0         11
25        2RB3WR           -34.43         8.10   6.49    29.87   -3.38                 2         11
27        3RB2WR           -41.33         8.16   3.25    28.57   -3.08                 1         11
35           5RB          


=== espn (1qb) — opening composition, best to worst (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1   1QB1RB2WR1TE            40.83         5.16  10.91    48.18    2.05                 7         11
2   1QB1RB1WR2TE            28.20         5.15  11.82    46.36    1.26                 5         11
3      1RB2WR2TE            27.28         5.10  11.82    44.55    1.47                 7         11
4      2RB2WR1TE            20.95         5.17  14.55    44.55    2.00                 8         11
5   1QB2RB1WR1TE            20.69         5.28   9.09    47.27    1.06                 6         11
15           ADP             0.00         5.50  10.00    40.00     NaN                 0         11
20        2RB3WR            -8.04         5.83   9.09    33.64   -1.34                 4         11
27        3RB2WR           -22.69         5.95  10.91    30.91   -2.96                 3         11
35        1QB4RB           -38.92  

In [11]:
# Where the full house actually lands, and what the control would have paid instead.
for league_key in ("sleeper", "espn"):
    table = ranking(league_key)
    place = {s: int(table.index[table.strategy == s][0]) for s in ("3RB2WR", "ADP")}
    full_house = table.loc[place["3RB2WR"]]
    print(
        f"{league_key:>8}: 3RB2WR ranks {place['3RB2WR']}/{len(table)} "
        f"({full_house.points_vs_field:+.1f} pts vs field, "
        f"{full_house.seasons_positive}/{full_house.n_seasons} seasons positive) — "
        f"the do-nothing ADP control ranks {place['ADP']}."
    )

 sleeper: 3RB2WR ranks 27/37 (-41.3 pts vs field, 1/11 seasons positive) — the do-nothing ADP control ranks 20.


    espn: 3RB2WR ranks 27/37 (-22.7 pts vs field, 3/11 seasons positive) — the do-nothing ADP control ranks 15.


<a id="significance"></a>
## 6. Is any of it significant?

Section 5 produced two orderings. This section asks whether they are signal, and **the answer now
differs by league** — which is itself the headline.

The t-tests run on the eleven per-season means rather than on the individual drafts behind them
(a few hundred per strategy, depending on league and field), because drafts within one season share
the same player outcomes: one running back tearing an ACL moves every
RB-heavy draft that year together. Treating those as independent would shrink the standard error by
roughly a factor of the number of draft slots and make almost everything "significant".

The first cell asks the **trend** question, which has more power than any single strategy: holding
everything else loose, does adding one more of a position to the opening help or hurt? Each cell is
the mean `points_vs_field` across every strategy with that many of that position; the slope is fitted
per season and t-tested across seasons.

- **ESPN (1QB): no position's slope clears significance.** Not running back, not receiver, not
  quarterback. What the cells show is an inverted U — zero of a position is bad, four or five is
  worse, the middle is flat. Opening composition there is worth about one good waiver claim, and the
  only genuinely costly openings are the extremes.
- **Sleeper (superflex): the quarterback slope is enormous and unambiguous** — the largest effect
  anywhere in this notebook, and the running back and receiver slopes now run *negative*, because in
  a 14-team superflex league every pick spent on a back is a pick not spent on the position that
  just gained a starting slot.

The second cell counts how many individual strategies clear |t| > 2 against how many chance alone
would produce (about two of thirty-seven). In ESPN the count is barely above chance and mostly on
the negative side — the sweep identifies losers, not winners. In Sleeper it is far above chance in
*both* directions: this is no longer a thin effect being read hopefully.

Two results worth pinning:

- **The full house is significantly bad in both leagues.** Not merely mid-table as it was under the
  old settings — below the bar, in each league independently. That is a cleaner verdict on the
  original question than this notebook has ever been able to give.
- **The opening that clears the bar in both leagues changed.** It used to be `2RB2WR1TE`, and under
  Sleeper's new settings that same opening is significantly *bad* there. The third cell recomputes
  the intersection rather than trusting the old answer, and what survives now takes a quarterback.

That second point is the reason this notebook derives `ACTUAL` from `league_settings` instead of
hard-coding it. The previous "robust in both leagues" finding was perfectly sound and is now
perfectly wrong, and nothing about the code would have told you.

In [12]:
compositions = q("""
    SELECT * FROM draft_strategy_results
    WHERE strategy_kind = 'composition' AND field_model = 'adp'
""")
for position in ("QB", "RB", "WR", "TE"):
    compositions[position] = (
        compositions.strategy.str.extract(rf"(\d){position}").fillna(0).astype(int)
    )


def trend(frame, position):
    """Mean vs-field by count, plus a season-clustered test of the slope."""
    per_season = frame.groupby(["season", position]).points_vs_field.mean().reset_index()
    slopes = [
        np.polyfit(group[position], group.points_vs_field, 1)[0]
        for _, group in per_season.groupby("season")
        if group[position].nunique() > 1
    ]
    t_stat, p_value = stats.ttest_1samp(slopes, 0.0)
    cells = frame.groupby(position).points_vs_field.mean()
    return {
        "position": position,
        **{f"{n} in opening": cells.get(n, np.nan) for n in range(6)},
        "pts/extra pick": np.mean(slopes),
        "t": t_stat,
        "p": p_value,
    }


for league_key in ("sleeper", "espn"):
    frame = compositions[
        (compositions.league_key == league_key)
        & (compositions.variant == ACTUAL[league_key])
    ]
    print(f"\n=== {league_key} / {ACTUAL[league_key]} (its real format) — "
          f"value of the Nth pick spent on a position ===")
    print(pd.DataFrame([trend(frame, p) for p in ("QB", "RB", "WR", "TE")])
          .set_index("position").round(2).to_string())


=== sleeper / superflex (its real format) — value of the Nth pick spent on a position ===
          0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
position                                                                                                                
QB              -58.36         29.17         71.90           NaN           NaN           NaN           65.13  6.46  0.00
RB              -12.80         32.70         21.67         -2.82        -43.39        -83.71          -17.35 -2.10  0.06
WR                5.11         20.78         19.14         -8.38        -42.19        -87.62          -19.43 -2.63  0.03
TE                7.37          8.92        -10.66           NaN           NaN           NaN           -9.02 -1.48  0.17

=== espn / 1qb (its real format) — value of the Nth pick spent on a position ===
          0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/ext

In [13]:
# Which strategies clear |t| > 2, and which way do they point?
significant = q("""
    SELECT league_key AS league, variant,
           COUNT(*) AS strategies,
           ROUND(0.05 * COUNT(*), 1) AS "expected by chance",
           SUM((ABS(t_stat) > 2)::INT) AS "|t|>2",
           SUM((t_stat > 2)::INT) AS good,
           SUM((t_stat < -2)::INT) AS bad
    FROM draft_strategy_summary
    WHERE field_model = 'adp' AND strategy_kind <> 'ordering'
    GROUP BY ALL ORDER BY league, variant
""")
print(significant.to_string(index=False))

named = q("""
    SELECT league_key, variant, t_stat > 2 AS good, strategy
    FROM draft_strategy_summary
    WHERE field_model = 'adp' AND strategy_kind <> 'ordering' AND ABS(t_stat) > 2
    ORDER BY league_key, variant, t_stat DESC
""")
for (league_key, variant), group in named.groupby(["league_key", "variant"]):
    for good in (True, False):
        names = group[group.good == good].strategy.tolist()
        if names:
            print(f"\n{league_key:>8} / {variant:<9} significantly "
                  f"{'GOOD' if good else 'BAD ':<4}: {', '.join(names)}")

 league   variant  strategies  expected by chance  |t|>2  good  bad
   espn       1qb          37                 1.9    7.0   2.0  5.0
   espn superflex          37                 1.9   15.0   8.0  7.0
sleeper       1qb          37                 1.9    5.0   0.0  5.0
sleeper superflex          37                 1.9   29.0  14.0 15.0



    espn / 1qb       significantly GOOD: 1QB1RB2WR1TE, 2RB2WR1TE

    espn / 1qb       significantly BAD : 5WR, 2QB3RB, 4RB1WR, 5RB, 3RB2WR

    espn / superflex significantly GOOD: 2QB1RB1WR1TE, 2QB2RB1WR, 1QB2RB2WR, 2QB1RB2WR, 1QB1RB2WR1TE, 2QB2RB1TE, 1QB2RB1WR1TE, 1QB1RB1WR2TE

    espn / superflex significantly BAD : 4RB1TE, 4RB1WR, 2RB3WR, 5WR, 1RB4WR, 5RB, 3RB2WR

 sleeper / 1qb       significantly BAD : 5RB, 4WR1TE, 3RB2TE, 5WR, 3WR2TE

 sleeper / superflex significantly GOOD: 2QB2RB1WR, 2QB1RB2WR, 1QB2RB1WR1TE, 1QB1RB2WR1TE, 1QB1RB3WR, 2QB1RB1WR1TE, 2QB3RB, 1QB2RB2WR, 1QB3RB1WR, 2QB2RB1TE, 2QB1RB2TE, 2QB2WR1TE, 1QB1RB1WR2TE, 1QB3RB1TE

 sleeper / superflex significantly BAD : 1RB4WR, 3RB1WR1TE, 3RB2WR, 4RB1TE, 1RB3WR1TE, 2RB3WR, 4RB1WR, 5WR, 5RB, 2RB2WR1TE, 3RB2TE, 4WR1TE, 2RB1WR2TE, 3WR2TE, 1RB2WR2TE


In [14]:
# The only claim that survives being asked of both leagues at once — each in its own real format.
actual = named[
    named.apply(lambda row: row.variant == ACTUAL[row.league_key], axis=1) & named.good
]
in_both = set.intersection(*(
    set(group.strategy) for _, group in actual.groupby("league_key")
)) if actual.league_key.nunique() == 2 else set()
print("openings significantly better than the field in BOTH leagues:",
      ", ".join(sorted(in_both)) or "none")

q("""
    SELECT s.league_key AS league, s.variant, s.strategy, s.points_vs_field, s.finish_rank,
           100 * s.top_third_rate AS "top3rd%", s.t_stat, s.seasons_positive, s.n_seasons
    FROM draft_strategy_summary s
    JOIN league_settings l ON l.league_key = s.league_key
    WHERE s.variant = CASE WHEN l.superflex_slots > 0 THEN 'superflex' ELSE '1qb' END
      AND s.field_model = 'adp' AND s.strategy = ?
    ORDER BY s.league_key
""", [sorted(in_both)[0]]).round(2) if in_both else None

openings significantly better than the field in BOTH leagues: 1QB1RB2WR1TE


,league,variant,strategy,points_vs_field,finish_rank,top3rd%,t_stat,seasons_positive,n_seasons
0,espn,1qb,1QB1RB2WR1TE,40.83,5.16,48.18,2.05,7,11
1,sleeper,superflex,1QB1RB2WR1TE,57.62,6.25,50.65,5.11,11,11


<a id="ordering"></a>
## 7. Ordering: RB early vs RB often

If composition matters, does the *order* within it? This is the last part of the full-house pitch
still standing, and it survives — but it has been demoted twice over.

Holding composition fixed and forcing an exact position sequence — every distinct permutation of
3RB+2WR and of 2RB+3WR — separates rosters that section 5 could not tell apart. One pattern holds
without exception in both leagues: **every ordering that opens with a running back beats every
ordering that opens with two receivers.** The third cell checks that rather than trusting the eye,
since "without exception" is exactly the kind of claim that quietly stops being true after a rebuild.

**Two caveats, and in Sleeper they are now larger than the finding.**

First, this is a relative claim. These forced sequences contain only backs and receivers, and in a
superflex league that constraint is close to disqualifying: *every* one of them finishes behind the
field in Sleeper, RB-first ones included. The second cell prints both groups, and the whole table
sits in negative territory. In ESPN the RB-first sequences are only mildly behind. So the finding is
*given* a back-and-receiver opening, take the back first — not that such an opening is anywhere you
want to be.

Second, the effect is smaller than the thing it sits inside. The third cell compares the spread
across orderings with the spread across sensible compositions; in Sleeper composition is now roughly
twice the lever ordering is. Getting the order right cannot rescue a composition that skips the
superflex.

Otherwise the same caveat as section 6 applies: individually these permutations mostly don't clear
|t| > 2. The claim worth making is the grouped one — opens-with-RB against opens-with-WR — which the
second cell tests directly.

In [15]:
orderings = q("""
    SELECT s.league_key, s.strategy, s.points_vs_field, s.finish_rank, s.t_stat,
           s.seasons_positive, s.n_seasons
    FROM draft_strategy_summary s
    JOIN league_settings l ON l.league_key = s.league_key
    WHERE s.strategy_kind = 'ordering' AND s.field_model = 'adp'
      AND s.variant = CASE WHEN l.superflex_slots > 0 THEN 'superflex' ELSE '1qb' END
    ORDER BY s.league_key, s.points_vs_field DESC
""")
orderings["opens"] = orderings.strategy.str.split("-").str[0]
orderings["RBs"] = orderings.strategy.str.count("RB")

for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key].drop(columns="league_key")
    print(f"\n=== {league_key} — forced opening sequences, best to worst ===")
    print(table.round(2).to_string(index=False))


=== sleeper — forced opening sequences, best to worst ===
      strategy  points_vs_field  finish_rank  t_stat  seasons_positive  n_seasons opens  RBs
RB-RB-WR-WR-WR           -15.97         7.93   -0.92                 3         11    RB    2
RB-WR-RB-WR-WR           -27.25         7.81   -4.15                 1         11    RB    2
RB-WR-WR-WR-RB           -28.93         8.05   -3.86                 2         11    RB    2
RB-RB-RB-WR-WR           -30.55         7.97   -1.92                 2         11    RB    3
RB-WR-WR-RB-WR           -30.58         8.03   -2.06                 3         11    RB    2
RB-RB-WR-WR-RB           -31.96         8.16   -1.94                 3         11    RB    3
RB-RB-WR-RB-WR           -34.96         8.21   -1.50                 3         11    RB    3
RB-WR-WR-RB-RB           -35.77         8.16   -2.91                 2         11    RB    3
WR-RB-RB-WR-WR           -38.81         8.18   -1.85                 2         11    WR    2
RB-WR-RB-WR

In [16]:
# The grouped claim: does opening with a back beat opening with a receiver?
rows = []
for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key]
    rb_first = table[table.opens == "RB"].points_vs_field
    wr_first = table[table.opens == "WR"].points_vs_field
    three_rb = table[table.RBs == 3].points_vs_field
    two_rb = table[table.RBs == 2].points_vs_field
    rows.append({
        "league": league_key,
        "opens RB": rb_first.mean(),
        "opens WR": wr_first.mean(),
        "RB-first edge": rb_first.mean() - wr_first.mean(),
        "3 RB total": three_rb.mean(),
        "2 RB total": two_rb.mean(),
        "3-RB edge": three_rb.mean() - two_rb.mean(),
    })
pd.DataFrame(rows).set_index("league").round(1)

,opens RB,opens WR,RB-first edge,3 RB total,2 RB total,3-RB edge
league,,,,,,
sleeper,-32.4,-53.3,20.9,-44.9,-40.8,-4.1
espn,-8.6,-23.3,14.7,-16.5,-15.4,-1.1


In [17]:
# "Every RB opener beats every WR-WR opener", checked. Also: how big is the ordering effect next to
# the composition effect it sits inside?
for league_key in ("sleeper", "espn"):
    table = orderings[orderings.league_key == league_key]
    rb_open = table[table.opens == "RB"].points_vs_field
    wr_wr_open = table[table.strategy.str.startswith("WR-WR")].points_vs_field
    holds = rb_open.min() > wr_wr_open.max()

    # "Sensible" = no all-in openings and no doubling up at QB or TE, i.e. the compositions a
    # human would actually be choosing between.
    all_compositions = ranking(league_key)
    sensible = all_compositions[
        ~all_compositions.strategy.str.contains(r"[45](?:RB|WR)|2QB|2TE")
    ].points_vs_field
    print(
        f"{league_key:>8}: worst RB-opener {rb_open.min():+.1f} vs best WR-WR opener "
        f"{wr_wr_open.max():+.1f} -> claim holds: {holds}\n"
        f"{'':>10}ordering spread {table.points_vs_field.max() - table.points_vs_field.min():.0f} pts; "
        f"spread across sensible compositions {sensible.max() - sensible.min():.0f} pts"
    )

 sleeper: worst RB-opener -48.5 vs best WR-WR opener -57.1 -> claim holds: True
          ordering spread 53 pts; spread across sensible compositions 105 pts


    espn: worst RB-opener -20.0 vs best WR-WR opener -27.1 -> claim holds: True
          ordering spread 44 pts; spread across sensible compositions 64 pts


In [18]:
orderings.groupby(["league_key", "opens"]).points_vs_field.agg(
    ["count", "mean", "min", "max"]
).round(1)

count  mean   min   max
league_key opens                         
espn       RB        10  -8.6 -20.0   3.2
           WR        10 -23.3 -40.5  -3.5
sleeper    RB        10 -32.4 -48.5 -16.0
           WR        10 -53.3 -68.9 -38.8

<a id="field"></a>
## 8. Who else is at the table

Every number so far assumes the other managers draft straight off ADP and never deviate. That is the
cleanest way to ask "does departing from the market pay", and it is also the friendliest, because a
lone deviator competes with nobody for the position it is hoarding.

`field_model = 'mixed'` drops that assumption: every opponent gets its own randomly drawn opening
composition, three seeded replicates per draft. The strategies keep their relative order — the rank
correlation between the two fields is high in both leagues — but the *level* moves.

**The headline here has reversed, and it is worth being explicit about why.** The previous version of
this notebook found that against a deviating field, plain best-available-by-ADP was the top plan
outright in Sleeper: when everyone else reaches to fill a quota they leave value on the board, and
the drafter with no quota to fill collects it. That was true of a 12-team, two-flex, 1QB league.

Under the current settings it is false, and badly so — the last cell tracks where the control lands
in each league and against each field. The mechanism is the one flagged in the intro: **the ADP board
is a 1QB board.** FantasyPros and FFC price for the formats their users mostly play, so
best-available off that board almost never takes a second quarterback. In ESPN that is fine, because
there is no second quarterback slot to fill. In Sleeper it means the disciplined drafter reliably
declines to fill the slot the league just added, and no amount of value collected elsewhere pays for
it.

So discipline is not a strategy-independent virtue; it is a bet that the board you are disciplined
about is priced for your league. In ESPN that bet is sound. In Sleeper the board is priced for a
different game, and following it faithfully is how you lose to people with worse process. Real
superflex ADP exists and would close most of this gap — this notebook doesn't have it, which is the
most actionable gap in the whole build.

In [19]:
agreement = []
for league_key in ("sleeper", "espn"):
    for variant in ("1qb", "superflex"):
        both = ranking(league_key, variant, "adp").merge(
            ranking(league_key, variant, "mixed"), on="strategy", suffixes=(" adp", " mixed")
        )
        agreement.append({
            "league": league_key,
            "variant": variant,
            "is real format": variant == ACTUAL[league_key],
            "strategies": len(both),
            "rank correlation": both["points_vs_field adp"].corr(
                both["points_vs_field mixed"], method="spearman"
            ),
        })
pd.DataFrame(agreement).set_index(["league", "variant"]).round(3)

is real format  strategies  rank correlation
league  variant                                                
sleeper 1qb                 False          37             0.856
        superflex            True          37             0.985
espn    1qb                  True          37             0.758
        superflex           False          37             0.841

In [20]:
for league_key in ("sleeper", "espn"):
    table = ranking(league_key, field_model="mixed")
    keep = pd.concat([
        table.head(5), table[table.strategy.isin(["ADP", "3RB2WR"])], table.tail(3)
    ]).drop_duplicates("strategy").sort_values("points_vs_field", ascending=False)
    print(f"\n=== {league_key} — against a field that also deviates (of {len(table)}) ===")
    print(keep.round(2).to_string())


=== sleeper — against a field that also deviates (of 37) ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB2RB1WR           110.66         5.43  15.37    55.63   13.77                11         11
2   2QB1RB1WR1TE           105.22         5.59  15.15    55.19    9.40                11         11
3      2QB1RB2WR           104.70         5.48  13.85    56.28   11.19                11         11
4      2QB2RB1TE            91.70         5.81  15.37    51.73    6.97                11         11
5      2QB1RB2TE            89.34         5.80  13.20    54.55    6.30                11         11
21           ADP            -9.39         7.72   8.66    32.47   -0.55                 5         11
25        3RB2WR           -39.59         8.27   4.33    26.84   -2.62                 2         11
35           5RB           -72.65         8.87   4.11    21.43   -2.64                 3         11
36        4WR1TE           -73.76     

In [21]:
# How far the do-nothing control moves once the rest of the room stops following ADP.
for league_key in ("sleeper", "espn"):
    moves = []
    for field_model in ("adp", "mixed"):
        table = ranking(league_key, field_model=field_model)
        moves.append((int(table.index[table.strategy == "ADP"][0]),
                      float(table.loc[table.strategy == "ADP", "points_vs_field"].iloc[0])))
    (adp_rank, adp_value), (mixed_rank, mixed_value) = moves
    print(f"{league_key:>8}: pure-ADP discipline ranks {adp_rank} vs a passive field "
          f"({adp_value:+.1f}) and {mixed_rank} vs a deviating one ({mixed_value:+.1f})")

 sleeper: pure-ADP discipline ranks 20 vs a passive field (-0.0) and 21 vs a deviating one (-9.4)
    espn: pure-ADP discipline ranks 15 vs a passive field (+0.0) and 18 vs a deviating one (+5.8)


<a id="superflex"></a>
## 9. What the superflex slot is doing

**Sleeper is superflex; ESPN is not.** The first cell checks that against the platforms' own settings
rather than asserting it: Sleeper's `roster_positions` now contains a `SUPER_FLEX` entry, and ESPN's
lineup slot 7 (`OP`, its superflex slot) is still 0.

Because every league is simulated in both formats, the slot can be priced directly: hold the league,
the board, the scoring and the roster size fixed, change only whether a quarterback is eligible for
one slot, and read the difference. `_lineup()` trades the slot against a *flex* spot, which is the
change a league actually makes when it goes superflex — exactly what Sleeper did — so starter count
and draft length are identical across the pair and the only moving part is quarterback eligibility.

The second cell puts the quarterback and running back slopes side by side across all four
league-format combinations, with the real format flagged. The pattern is the same in both leagues and
does not care which one is real: **add the slot and the quarterback slope goes from indistinguishable
from zero to the largest effect in the notebook; remove it and the effect vanishes.** That is about as
clean an identification as this data allows, and it means Sleeper's result is a fact about superflex
rather than a fact about Sleeper.

It also answers the counterfactual in the other direction, which is now the useful one for ESPN: if
that league ever adds the slot, everything in sections 5-8 changes for it the way it just changed for
Sleeper, and this section already has the numbers.

One caveat on the magnitude, and it points the same way in both leagues. The ADP board is a 1QB
board, so a focal team drafting a superflex lineup buys quarterbacks at one-quarterback prices — real
superflex ADP has already repriced exactly that. Read the superflex quarterback figures as an **upper
bound** on what an early quarterback is worth, and the gap between the two formats as what the slot
itself is doing. The direction is not in doubt; the size is.

In [22]:
# Which league has the slot, read from the platforms' own settings rather than asserted.
# This is the check whose answer changed in August 2026.
sf = q("SELECT league_key, superflex_slots, qb_slots FROM league_settings ORDER BY league_key")
for _, row in sf.iterrows():
    verdict = (
        f"{row.superflex_slots} superflex slot(s) — QB is flex-eligible"
        if row.superflex_slots
        else f"no superflex slot: {row.qb_slots} starting QB, and QB cannot fill a flex spot"
    )
    print(f"{row.league_key:>8}: {verdict}")

print(
    "\nsuperflex leagues among mine:",
    int((sf.superflex_slots > 0).sum()), "of", len(sf),
)

    espn: no superflex slot: 1 starting QB, and QB cannot fill a flex spot
 sleeper: 1 superflex slot(s) — QB is flex-eligible

superflex leagues among mine: 1 of 2


In [23]:
for position in ("QB", "RB"):
    rows = []
    for league_key in ("sleeper", "espn"):
        for variant in ("1qb", "superflex"):
            frame = compositions[
                (compositions.league_key == league_key)
                & (compositions.variant == variant)
            ]
            rows.append({
                "league": league_key,
                "variant": variant,
                "is real format": variant == ACTUAL[league_key],
                **trend(frame, position),
            })
    print(f"\n=== value of the Nth {position} in the opening, 1QB vs superflex ===")
    print(pd.DataFrame(rows).drop(columns="position")
          .set_index(["league", "variant"]).round(2).to_string())


=== value of the Nth QB in the opening, 1QB vs superflex ===
                   is real format  0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
league  variant                                                                                                                                  
sleeper 1qb                 False        -23.50        -12.10        -19.20           NaN           NaN           NaN            2.15  0.25  0.81
        superflex            True        -58.36         29.17         71.90           NaN           NaN           NaN           65.13  6.46  0.00
espn    1qb                  True        -14.39          3.39        -12.84           NaN           NaN           NaN            0.77  0.11  0.91
        superflex           False        -26.11         18.62         34.61           NaN           NaN           NaN           30.36  7.56  0.00



=== value of the Nth RB in the opening, 1QB vs superflex ===
                   is real format  0 in opening  1 in opening  2 in opening  3 in opening  4 in opening  5 in opening  pts/extra pick     t     p
league  variant                                                                                                                                  
sleeper 1qb                 False        -38.44         -1.28         -6.58        -18.42        -32.36        -52.63           -5.03 -0.55  0.59
        superflex            True        -12.80         32.70         21.67         -2.82        -43.39        -83.71          -17.35 -2.10  0.06
espn    1qb                  True        -23.15         13.70          7.89        -15.81        -36.47        -64.43          -10.87 -1.13  0.29
        superflex           False        -19.05         32.51         29.60         -3.09        -39.64        -77.08          -15.41 -1.69  0.12


In [24]:
for league_key in ("sleeper", "espn"):
    for variant in ("1qb", "superflex"):
        table = ranking(league_key, variant, "mixed")
        keep = pd.concat([table.head(5), table[table.strategy == "ADP"]]).drop_duplicates("strategy")
        label = "REAL FORMAT" if variant == ACTUAL[league_key] else "counterfactual"
        print(f"\n=== {league_key} / {variant} ({label}) — best openings vs a deviating field ===")
        print(keep.round(2).to_string())


=== sleeper / 1qb (counterfactual) — best openings vs a deviating field ===
       strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1     1QB1RB3WR            44.65         6.61   8.66    44.81    3.89                10         11
2     1QB2RB2WR            41.79         6.69   8.44    42.64    4.97                11         11
3     1QB3RB1WR            37.45         6.73   8.01    44.59    3.68                10         11
4  1QB2RB1WR1TE            36.01         6.87  10.17    39.61    3.48                 9         11
5  1QB1RB2WR1TE            29.75         6.93   8.66    40.04    2.63                 9         11
6           ADP            24.66         7.00  11.69    40.04    2.30                 8         11

=== sleeper / superflex (REAL FORMAT) — best openings vs a deviating field ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      2QB2RB1WR           110.66         5.43  15


=== espn / 1qb (REAL FORMAT) — best openings vs a deviating field ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1      1QB2RB2WR            36.71         5.02  14.55    44.24    3.37                 8         11
2      1QB3RB1WR            30.07         5.07  15.45    46.67    2.36                 8         11
3   1QB1RB2WR1TE            29.50         5.00  11.82    46.06    2.25                10         11
4   1QB1RB1WR2TE            28.78         5.02  12.42    48.18    2.28                 8         11
5   1QB2RB1WR1TE            25.10         5.06  14.55    48.48    2.16                 8         11
18           ADP             5.76         5.52  11.52    39.39    0.30                 5         11

=== espn / superflex (counterfactual) — best openings vs a deviating field ===
        strategy  points_vs_field  finish_rank   win%  top3rd%  t_stat  seasons_positive  n_seasons
1   2QB1RB1WR1TE            71.76         4.42  1

<a id="doing"></a>
## 10. What to actually do

The two leagues need different plans now, so this is per league, in descending order of how much the
notebook believes each claim. The cell below assembles the same recommendation from the tables rather
than from this list, so a rebuild that moves the numbers moves it here too.

**Sleeper — 14 teams, superflex**

1. **Take two quarterbacks in the first five rounds.** The quarterback slope is the largest and most
   significant effect anywhere in this notebook, every opening in the top ten takes at least one and
   the top seven take two, and the result is positive in essentially every season (§6, §9).
2. **Do not draft best-available off the consensus board.** It is a 1QB board and it will not take
   the second quarterback for you; the do-nothing control is now a below-average plan in this league
   against either field (§8). This is the single biggest change from the previous version.
3. **Don't run the full house, and don't run its balanced cousin either.** `3RB2WR` is significantly
   bad here, and so is `2RB2WR1TE`, which was this notebook's headline recommendation under the old
   settings (§6).
4. **Running back scarcity is real here but comprehensively outranked.** The premium is genuine and
   deeper than in ESPN (§2), and it is still the wrong place to spend the first five rounds.

**ESPN — 10 teams, 1QB**

1. **Composition barely matters; avoid the extremes.** No position's slope clears significance (§6).
   Four or five of one position is the mistake, zero is also bad, and the middle is flat.
2. **`1QB1RB2WR1TE` is the one opening that clears the bar in both leagues** (§6) — worth preferring
   not because it is optimal but because it is the least format-dependent thing here.
3. **Don't run the full house.** `3RB2WR` is significantly bad in ESPN too, and it spends its third
   premium pick in rounds 3-5 where backs return least and hit least often (§3, §4, §6).
4. **Ignore running back scarcity arguments.** In full PPR with a shallow pool the premium is nearly
   gone by the fourth back off the board (§2).
5. **If ESPN ever adds a superflex slot, throw this list out** and read section 9 — the answer
   changes completely, and quickly.

**Both leagues:** within a back-and-receiver opening, take the back first (§7) — a tiebreaker on
ordering, not a reason to build that opening.

In [25]:
for league_key in ("sleeper", "espn"):
    settings = q("SELECT * FROM league_settings WHERE league_key = ?", [league_key]).iloc[0]
    passive = ranking(league_key, field_model="adp")
    deviating = ranking(league_key, field_model="mixed")
    best = passive.iloc[0]
    rb_first = orderings[(orderings.league_key == league_key) & (orderings.opens == "RB")]
    wr_first = orderings[(orderings.league_key == league_key) & (orderings.opens == "WR")]

    print(f"\n{'=' * 68}\n{league_key.upper()} — {settings.team_count} teams, "
          f"{settings.rec_pts} PPR, {settings.flex_slots} flex, "
          f"{settings.superflex_slots} superflex\n{'=' * 68}")
    print(f"  best opening vs a passive field : {best.strategy} "
          f"({best.points_vs_field:+.0f} pts, t={best.t_stat:.2f})")
    print(f"  best opening vs a deviating field: {deviating.iloc[0].strategy} "
          f"({deviating.iloc[0].points_vs_field:+.0f} pts, t={deviating.iloc[0].t_stat:.2f})")
    print(f"  full house (3RB2WR)             : rank {int(passive.index[passive.strategy == '3RB2WR'][0])}"
          f" of {len(passive)}")
    print(f"  open RB vs open WR              : "
          f"{rb_first.points_vs_field.mean() - wr_first.points_vs_field.mean():+.0f} pts for RB first")
    print(f"  QB in the first five rounds     : "
          f"{'yes — superflex' if settings.superflex_slots else 'no — one QB slot, no flex eligibility'}")


SLEEPER — 14 teams, 0.5 PPR, 1 flex, 1 superflex
  best opening vs a passive field : 2QB1RB2WR (+108 pts, t=6.93)
  best opening vs a deviating field: 2QB2RB1WR (+111 pts, t=13.77)
  full house (3RB2WR)             : rank 27 of 37
  open RB vs open WR              : +21 pts for RB first
  QB in the first five rounds     : yes — superflex



ESPN — 10 teams, 1.0 PPR, 1 flex, 0 superflex
  best opening vs a passive field : 1QB1RB2WR1TE (+41 pts, t=2.05)
  best opening vs a deviating field: 1QB2RB2WR (+37 pts, t=3.37)
  full house (3RB2WR)             : rank 27 of 37
  open RB vs open WR              : +15 pts for RB first
  QB in the first five rounds     : no — one QB slot, no flex eligibility


<a id="board"></a>
## 11. The 2026 board

Strategy decides *which position* to take; `draft_value` decides which player, by pricing this year's
projection against what that ADP slot has historically returned. `projected_surplus_rank` is the
draft-board column — at any given pick, who is expected to return the most over what he costs.

Shown per position so it can be read the way a draft actually goes: you are picking within a position
tier, not off one global list.

Two things to hold in mind while reading it, both from section 8. The ADP it prices against is a 1QB
board, so in Sleeper the quarterback surpluses here are if anything *understated* relative to what
that league's superflex slot is worth. And surplus ranks players against their cost, which is a
different question from which position to spend the pick on — section 10 answers that, this table
answers "given that, who".

In [26]:
live_season = q("SELECT MAX(season) AS s FROM draft_value").s.iloc[0]
board = q("""
    SELECT position, player_name, ROUND(consensus_adp, 1) AS adp,
           ROUND(projected_surplus, 1) AS surplus, CAST(projected_surplus_rank AS INT) AS rank
    FROM draft_value
    WHERE league_key = 'sleeper' AND season = ? AND projected_surplus IS NOT NULL
      AND consensus_adp <= 120
    QUALIFY ROW_NUMBER() OVER (PARTITION BY position ORDER BY projected_surplus DESC) <= 8
    ORDER BY position, projected_surplus DESC
""", [int(live_season)])
print(f"{live_season} — best value per position inside the first 10 rounds (Sleeper scoring)")
board

2026 — best value per position inside the first 10 rounds (Sleeper scoring)


,position,player_name,adp,surplus,rank
0,QB,Jalen Hurts,69.9,32.5,3
1,QB,Josh Allen,27.4,9.8,30
2,QB,Bo Nix,109.4,7.4,31
3,QB,Caleb Williams,82.8,2.7,37
4,QB,Jaxson Dart,103.5,-0.8,58
5,QB,Justin Herbert,91.5,-5.7,124
6,QB,Patrick Mahomes,102.2,-17.1,225
7,QB,Jared Goff,107.9,-31.0,252
8,RB,Jeremiyah Love,26.7,57.1,1
9,RB,D'Andre Swift,48.5,32.5,4


---

## Adding to this notebook

The simulator lives in `src/gold/draft_strategy.py`, not here — its docstring carries the full method
and the caveats, and `python -m src.gold.draft_strategy` reprints the summary tables. This notebook
only reads `draft_strategy_results` / `draft_strategy_summary` and the models they were built from.

`q()` opens a read-only connection, runs, and closes it — so nothing here can hold a lock that blocks
`scripts/build_warehouse.sh`, and nothing here can write to the warehouse. See `notebooks/README.md`
for the conventions.

Useful while exploring:

```python
tables("gold")
columns("draft_strategy_summary")
peek("draft_strategy_results")
```

**Derive league facts, don't assert them.** This notebook previously hard-coded "neither league is
superflex" into a dozen queries. Sleeper changed one slot and every one of them silently read the
wrong rows — no error, no failed cell, just a set of confident conclusions about a league that no
longer existed. `ACTUAL` in section 1 is the fix; anything else that depends on settings should be
read the same way.

Worth testing if this gets picked up again:

- **A real superflex ADP board.** The biggest known bias in the build (§8, §9): quarterbacks are
  priced for 1QB leagues throughout, which makes Sleeper's superflex numbers an upper bound and makes
  the pure-ADP control look worse than a well-priced board would.
- **Keeper/dynasty rules.** Everything here is a redraft simulation.
- **In-season roster churn.** The simulation drafts and then freezes; it has no waiver wire, and a
  strategy that leaves you thin at a position is punished less here than in a real season.
- **Weekly lineups rather than season totals.** Teams are scored on the hindsight-best lineup, which
  is the same fiction for every strategy but flatters high-variance rosters slightly.